# Notebook 09-C — SVM Mejorado: Texto Sin Procesar (Sin Remocion de Stopwords ni Lematizacion)

## Objetivo
Replicar el hallazgo de investigadores que reportaron **F1 = 0.9581** con un
SVM lineal entrenado sobre **textos sin procesar** — es decir, conservando
las *stopwords*, la puntuacion y las mayusculas originales.

### Fundamentacion (articulo de referencia)

> *"La metrica del 95.81% es reportada por el modelo SVM lineal\*.*
> *El asterisco (\*) denota que el algoritmo fue entrenado utilizando*
> *textos sin procesar. Esto significa que [...] decidieron no eliminar*
> *las palabras vacias (stopwords) y no aplicar lematizacion."*

En nuestro pipeline, esto equivale a usar la columna **`texto_beto`** en lugar de
`texto_lineal`. La diferencia es exactamente la que describe el articulo:

| Columna | Stopwords | Puntuacion | Mayusculas | Limpieza |
|---|---|---|---|---|
| `texto_lineal` (NB-09) | Eliminadas | Eliminada | Minusculas | Estricta |
| `texto_beto` (este NB) | **Conservadas** | **Conservada** | **Conservadas** | Solo URLs/HTML |

### Hiperparametros
Identicos al NB-09 original (solo cambia la columna de texto):
- `LinearSVC(max_iter=2000, random_state=42)`
- `max_features`: [5000, 10000]
- `C`: [0.1, 1.0, 10.0]
- `StratifiedKFold(K=5, shuffle=True, random_state=42)`
- `RandomUnderSampler(random_state=42)`
- Metrica: `f1_macro`

### Formato de salida esperado
```
SVM (texto sin procesar): F1-Score: 0.XX | Tiempo: XX min | RAM: X.XX GB
```

## Celda 0 — Verificacion de memory_profiler

In [2]:
import memory_profiler
print(f"memory_profiler version: {memory_profiler.__version__}")

memory_profiler version: 0.61.0


## Celda 1 — Importaciones

In [3]:
import pandas as pd
import time
import warnings
import gc

# Monitoreo de RAM
from memory_profiler import memory_usage

# scikit-learn
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.svm import LinearSVC
from sklearn.base import clone

# imbalanced-learn
from imblearn.pipeline import Pipeline
from imblearn.under_sampling import RandomUnderSampler

warnings.filterwarnings('ignore')
print("Librerias cargadas correctamente.")

Librerias cargadas correctamente.


## Celda 2 — Carga del Corpus: columna `texto_beto` (texto sin procesar)

La columna `texto_beto` contiene el texto con **limpieza minima** (solo se
removieron URLs y etiquetas HTML). Conserva:
- Mayusculas originales
- Signos de puntuacion
- Palabras vacias (stopwords)

Esto es exactamente lo que el articulo llama "textos sin procesar".

In [4]:
INPUT_PATH = 'data/train_val_set.csv'

df = pd.read_csv(INPUT_PATH, sep=';')
df['texto_beto'] = df['texto_beto'].fillna('')

# CAMBIO CLAVE: usamos texto_beto en lugar de texto_lineal
X = df[['texto_beto']]
y = df['label']

print("Corpus cargado.")
print(f"   Total de registros : {len(df):,}")
print(f"   Columna de texto   : texto_beto (sin procesar)")
print(f"   Distribucion de clases:\n{y.value_counts().to_string()}")

print("\n--- Ejemplo de texto ---")
print(f"   texto_beto   : {df['texto_beto'].iloc[0][:120]}...")
print(f"   texto_lineal : {df['texto_lineal'].iloc[0][:120]}...")
print("\n   Diferencia: texto_beto conserva stopwords, puntuacion y mayusculas.")

Corpus cargado.
   Total de registros : 45,784
   Columna de texto   : texto_beto (sin procesar)
   Distribucion de clases:
label
1    26680
0    19104

--- Ejemplo de texto ---
   texto_beto   : Las bases de Bcomú apoyan por gran mayoría un pacto con el PSC y los votos de Valls para investir a Ada Colau como alcal...
   texto_lineal : bases bcomú apoyan gran mayoría pacto psc votos valls investir ada colau alcaldesa resultado consulta saldado votos favo...

   Diferencia: texto_beto conserva stopwords, puntuacion y mayusculas.


## Celda 3 — Pipeline y Validacion Cruzada Estratificada

Pipeline identico al NB-09 pero con `texto_beto` en el `ColumnTransformer`.
El `TfidfVectorizer` por defecto **no** elimina stopwords (`stop_words=None`),
asi que el flujo es coherente con el articulo.

In [5]:
# Validacion Cruzada Estratificada (K=5) — misma semilla que NB-09
cv_estratificado = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Submuestreo para balance de clases (identico al NB-09)
undersampler = RandomUnderSampler(random_state=42)

# Vectorizador TF-IDF sobre 'texto_beto' (el cambio clave)
# stop_words=None (default) — NO se eliminan stopwords
preprocessor = ColumnTransformer(
    transformers=[('tfidf', TfidfVectorizer(), 'texto_beto')]
)

# Pipeline: undersampler -> TF-IDF -> clasificador
pipeline_base = Pipeline(steps=[
    ('undersampler', undersampler),
    ('vectorizer', preprocessor),
    ('classifier', LinearSVC())  # placeholder
])

print("Pipeline configurado.")
print(f"  Columna de texto : texto_beto (sin procesar)")
print(f"  Stopwords        : conservadas (TfidfVectorizer default)")
print(f"  Lematizacion     : no aplicada")
print(f"  CV               : StratifiedKFold(K=5, random_state=42)")
print(f"  Balanceo         : RandomUnderSampler(random_state=42)")

Pipeline configurado.
  Columna de texto : texto_beto (sin procesar)
  Stopwords        : conservadas (TfidfVectorizer default)
  Lematizacion     : no aplicada
  CV               : StratifiedKFold(K=5, random_state=42)
  Balanceo         : RandomUnderSampler(random_state=42)


## Celda 4 — Helper: funcion de entrenamiento + medicion de RAM

In [6]:
def entrenar_con_metricas(grid_search_obj, X_data, y_data, nombre_modelo):
    """
    Ejecuta grid_search_obj.fit(X_data, y_data) midiendo:
      - Pico de RAM (memory_profiler) en GB
      - Tiempo total en minutos
    Retorna: (best_score, tiempo_min, ram_gb)
    """
    print(f"\n{'='*60}")
    print(f"  Entrenando: {nombre_modelo}")
    print(f"  (GridSearchCV K=5 — texto sin procesar)")
    print(f"{'='*60}")

    def _fit():
        grid_search_obj.fit(X_data, y_data)

    t_inicio = time.time()

    muestras_ram = memory_usage(
        (_fit, [], {}),
        interval=0.1,
        include_children=True,
        max_usage=False,
        retval=False
    )

    t_fin = time.time()

    pico_ram_mb = max(muestras_ram)
    pico_ram_gb = pico_ram_mb / 1024
    tiempo_min  = (t_fin - t_inicio) / 60
    best_score  = grid_search_obj.best_score_

    print(f"\n  Entrenamiento finalizado.")
    print(f"  Mejor configuracion : {grid_search_obj.best_params_}")
    print(f"  Mejor F1-Score Macro: {best_score:.4f}")
    return best_score, tiempo_min, pico_ram_gb

## Celda 5 — GridSearchCV para SVM sobre Texto Sin Procesar

Mismos hiperparametros que el NB-09 original:
- `max_features`: [5000, 10000]
- `C`: [0.1, 1.0, 10.0]
- 6 combinaciones x 5 folds = 30 fits

In [7]:
# Malla identica al NB-09
param_grid_svm = [
    {
        'vectorizer__tfidf__max_features': [5000, 10000],
        'classifier': [LinearSVC(max_iter=2000, random_state=42)],
        'classifier__C': [0.1, 1.0, 10.0]
    }
]

grid_svm = GridSearchCV(
    estimator=clone(pipeline_base),
    param_grid=param_grid_svm,
    cv=cv_estratificado,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1,
    refit=True
)

f1_svm, tiempo_svm, ram_svm = entrenar_con_metricas(
    grid_svm, X, y, "SVM — LinearSVC (texto sin procesar)"
)

gc.collect()
print("\nMemoria liberada.")


  Entrenando: SVM — LinearSVC (texto sin procesar)
  (GridSearchCV K=5 — texto sin procesar)
Fitting 5 folds for each of 6 candidates, totalling 30 fits

  Entrenamiento finalizado.
  Mejor configuracion : {'classifier': LinearSVC(max_iter=2000, random_state=42), 'classifier__C': 1.0, 'vectorizer__tfidf__max_features': 10000}
  Mejor F1-Score Macro: 0.9034

Memoria liberada.


## Celda 6 — Ranking Completo de Combinaciones

In [8]:
import numpy as np

resultados_cv = pd.DataFrame(grid_svm.cv_results_)

# Extraer columnas relevantes
cols = [c for c in resultados_cv.columns
        if c.startswith('param_') or c in ['mean_test_score', 'std_test_score', 'rank_test_score']]
ranking = resultados_cv[cols].sort_values('rank_test_score')

# Renombrar para legibilidad
rename_map = {}
for c in ranking.columns:
    if 'max_features' in c:
        rename_map[c] = 'max_features'
    elif 'classifier__C' in c:
        rename_map[c] = 'C'
    elif c == 'mean_test_score':
        rename_map[c] = 'F1 Macro (mean)'
    elif c == 'std_test_score':
        rename_map[c] = 'F1 Macro (std)'
    elif c == 'rank_test_score':
        rename_map[c] = 'Rank'

ranking = ranking.rename(columns=rename_map)
# Eliminar columna del clasificador (misma para todas)
ranking = ranking.drop(
    columns=[c for c in ranking.columns if 'classifier' in c.lower() and c not in rename_map.values()],
    errors='ignore'
)

print(f"\n{'='*60}")
print(f"  Ranking: SVM con texto sin procesar")
print(f"{'='*60}")
display(ranking.reset_index(drop=True))


  Ranking: SVM con texto sin procesar


,C,max_features,F1 Macro (mean),F1 Macro (std),Rank
0,1.0,10000,0.903368,0.001871,1
1,1.0,5000,0.903232,0.002181,2
2,0.1,5000,0.898995,0.001924,3
3,0.1,10000,0.897675,0.001353,4
4,10.0,5000,0.884121,0.001337,5
5,10.0,10000,0.876565,0.001470,6


## Celda 7 — Tabla Comparativa: NB-09 (texto procesado) vs NB-09-C (texto sin procesar)

In [9]:
print("\n" + "=" * 75)
print("  RESULTADOS — SVM con Texto Sin Procesar (segun articulo de referencia)")
print("=" * 75)
print(f"  SVM (texto sin procesar): F1-Score: {f1_svm:.2f} | Tiempo: {tiempo_svm:.1f} min | RAM: {ram_svm:.2f} GB")
print("=" * 75)

# Resultados NB-09 original (del output ya registrado)
comparativa = pd.DataFrame({
    'Notebook':           ['09 (original)', '09-C (este)'],
    'Columna de texto':   ['texto_lineal', 'texto_beto'],
    'Stopwords':          ['Eliminadas', 'Conservadas'],
    'Puntuacion':         ['Eliminada', 'Conservada'],
    'Mayusculas':         ['Minusculas', 'Conservadas'],
    'F1-Score Macro':     [0.9035, round(f1_svm, 4)],
    'Tiempo (min)':       [0.48, round(tiempo_svm, 2)],
    'RAM Pico (GB)':      [2.052, round(ram_svm, 3)],
    'Mejor C':            [1.0, grid_svm.best_params_.get('classifier__C', 'N/A')],
    'max_features':       [5000, grid_svm.best_params_.get('vectorizer__tfidf__max_features', 'N/A')]
})

# Calcular delta
delta_f1 = round(f1_svm - 0.9035, 4)
signo = '+' if delta_f1 >= 0 else ''

print(f"\n  Delta F1 (sin procesar - procesado): {signo}{delta_f1}")
if delta_f1 > 0:
    print(f"  El texto sin procesar MEJORA el F1-Score.")
elif delta_f1 < 0:
    print(f"  El texto sin procesar EMPEORA el F1-Score.")
else:
    print(f"  Sin diferencia significativa.")

print("\n  Tabla comparativa:")
display(comparativa.set_index('Notebook'))


  RESULTADOS — SVM con Texto Sin Procesar (segun articulo de referencia)
  SVM (texto sin procesar): F1-Score: 0.90 | Tiempo: 0.7 min | RAM: 2.17 GB

  Delta F1 (sin procesar - procesado): -0.0001
  El texto sin procesar EMPEORA el F1-Score.

  Tabla comparativa:


,Columna de texto,Stopwords,Puntuacion,Mayusculas,F1-Score Macro,Tiempo (min),RAM Pico (GB),Mejor C,max_features
Notebook,,,,,,,,,
09 (original),texto_lineal,Eliminadas,Eliminada,Minusculas,0.9035,0.48,2.052,1.0,5000
09-C (este),texto_beto,Conservadas,Conservada,Conservadas,0.9034,0.72,2.169,1.0,10000


## Celda 8 — Exportar Resultados

In [10]:
import joblib

# Resultados CSV
resumen = pd.DataFrame({
    'Modelo':          ['SVM (LinearSVC)'],
    'Texto':           ['sin procesar (texto_beto)'],
    'F1-Score Macro':  [round(f1_svm, 4)],
    'Tiempo (min)':    [round(tiempo_svm, 2)],
    'RAM Pico (GB)':   [round(ram_svm, 3)],
    'Mejor C':         [grid_svm.best_params_.get('classifier__C', 'N/A')],
    'max_features':    [grid_svm.best_params_.get('vectorizer__tfidf__max_features', 'N/A')],
    'Stopwords':       ['conservadas'],
    'Lematizacion':    ['no']
})

resumen.to_csv('data/resultados_svm_texto_sin_procesar.csv', index=False, sep=';')
print("Resultados guardados en: data/resultados_svm_texto_sin_procesar.csv")

# Ranking CV completo
cv_df = pd.DataFrame(grid_svm.cv_results_)
cv_df.to_csv('data/cv_results_svm_texto_sin_procesar.csv', index=False, sep=';')
print("Ranking CV guardado en : data/cv_results_svm_texto_sin_procesar.csv")

# Modelo entrenado (refit=True -> ajustado al corpus completo)
joblib.dump(grid_svm.best_estimator_, 'data/best_svm_texto_sin_procesar.joblib')
print("Modelo best_estimator_ : data/best_svm_texto_sin_procesar.joblib")

print("\nExportacion completa.")

Resultados guardados en: data/resultados_svm_texto_sin_procesar.csv
Ranking CV guardado en : data/cv_results_svm_texto_sin_procesar.csv
Modelo best_estimator_ : data/best_svm_texto_sin_procesar.joblib

Exportacion completa.
